# 🔬 29 · Sharpening the moving parts: GMM-based focused refinement
### Estimate how a domain moved in each particle, then re-align the data on that domain — in single particles and in cells

**BiologyToResearch** · Part F — Conformational heterogeneity: Gaussian mixtures and neural fields · Module 29 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/29_gmm_focused_refinement.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 4 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, PyTorch |
| **Prerequisites** | [25 · Gaussian mixtures](25_gaussian_mixture_molecules.ipynb), [26 · e2gmm](26_e2gmm_heterogeneity.ipynb) (rigid-body decoders), [28 · tomoDRGN](28_tomodrgn_subtilts.ipynb) (sub-tilts), [10 · Local resolution](10_resolution_validation.ipynb) |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Chen 2026 (GMM-based focused refinement for cryo-EM and cryo-ET, arXiv); Chen, Schmid & Chiu 2024 (improving resolution and resolvability with GMMs, *Nat Methods*) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods (1,500 particles of 40 × 40 pixels at 4 Å, a
> 200-Gaussian model, one round of refinement). They illustrate the core ideas; they do **not** reproduce the papers'
> numbers. **All molecules and images are generated by code** (nothing is downloaded); no CTF. Because we simulate the
> data we can also measure the *true* motions and the *true* map — something real projects never have.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why a domain that moves from particle to particle is blurred in a consensus map even when every particle's
   overall pose is right, and **measure** the blur region by region.
2. **Implement** GMM-based focused refinement: rotate only the Gaussians of a chosen region, separately for each
   particle, to maximise the FRC with that particle (direct gradient optimisation).
3. **Turn** each particle's estimated domain motion into a **new pose** for that domain, reconstruct with the new poses,
   and **combine** region maps into a **composite** map.
4. **Explain** the tricks of Chen, Schmid & Chiu (2024): converting conformations into poses, patch-by-patch refinement,
   and widening the Gaussians outside the patch.
5. **Extend** the method to cryo-ET sub-tilts: one domain motion per 3D particle, scored over all its tilts.
6. **Recognise** the method's caveats: noise-limited per-particle estimates, priors, and why a better map of a flexible
   domain can *lower* a global FSC.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools, PyTorch helpers | 5 min |
| 1 · A wobbling domain | simulate 1,500 particles whose left half wobbles | 10 min |
| 2 · What the consensus does to it | region-by-region quality of the consensus map | 10 min |
| 3 · A Gaussian model and a focus mask | GMM of the consensus, Gaussians of the region | 10 min |
| 4 · Focused refinement | per-particle rotations by gradient descent | 25 min |
| 5 · From motions to poses to maps | new poses, focused reconstruction, composite map | 20 min |
| 6 · Patches and the 2024 tricks | conformation → pose, patch-by-patch, widening | 10 min |
| 7 · Inside cells: sub-tilts | one motion per particle, FRC over tilts | 20 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

The TRPV1 ion channel senses heat and the chilli compound capsaicin. Its transmembrane core is rigid enough to be solved at
near-atomic resolution, but its large cytoplasmic domains flex, and in a consensus map they come out blurred — exactly
where regulatory proteins and drugs bind. ATP synthase, the rotary motor that makes most of a cell's ATP, is worse: its
head rotates relative to its stalk as it works. Classical "focused refinement" re-aligns particles on a masked region,
but with small regions and noisy images that alignment is unreliable. Muyuan Chen's GMM approach (2024, 2026) uses the
whole molecule's Gaussian model to estimate how the region moved in each particle — then lets the data, re-aligned on
that region, speak for themselves.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

Helpers copied from [`course_toolkit.py`](../course_toolkit.py): the Part F protein, the Gaussian-mixture helpers,
`pseudoatoms_kmeans` and `reconstruct_fourier` (Module 25), rotations (Module 02), `lowpass` (Module 03).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def rot_y(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def flexible_protein(seed: int = 7):
    """A code-generated ~80 kDa protein for heterogeneity experiments (Modules 25–30).

    Domains 0 and 1 form a rigid body; domain 2 (the 'arm') sits on domain 1 and can swing about a hinge; domain 3 is
    a small partner subunit that may be bound or not. Returns (atoms, hinge (x, y, z) in Å); atoms['domain'] labels
    every atom. Coordinates are in Å, centred near the origin.
    """
    rng = np.random.default_rng(seed)
    specs = [((21.0, 17.5, 16.0), (-19.0, -7.5, 0.0)), ((19.0, 16.0, 17.5), (19.0, -7.5, 0.0)),
             ((17.5, 12.5, 14.0), (21.0, 33.0, 0.0)), ((12.5, 11.0, 11.0), (-21.0, 26.0, 2.5))]
    parts = []
    for d, (radii, centre) in enumerate(specs):
        part = _transform(helix_bundle(radii, rng, chain="ABCD"[d]), _rz(rng.uniform(0, 2 * np.pi)), centre)
        part["domain"] = np.full(len(part["xyz"]), d)
        parts.append(part)
    atoms = merge_atoms(parts)
    shift = atoms["xyz"][atoms["domain"] < 3].mean(0)
    atoms["xyz"] = (atoms["xyz"] - shift).astype(np.float32)
    return atoms, np.array([19.0, 14.0, 0.0]) - shift


def pseudoatoms_kmeans(vol: np.ndarray, n: int, voxel: float = 1.0, rng=None, iters: int = 30,
                       level: float = 0.05):
    """Summarise a density map by n Gaussian 'pseudo-atoms': weighted K-means on the voxels above level × max.

    Returns centres (n, 3) as (x, y, z) in Å relative to the box centre, and amplitudes (density in each cluster).
    """
    rng = np.random.default_rng(0) if rng is None else rng
    idx = np.argwhere(vol > level * vol.max())                                 # (z, y, x) of dense voxels
    w = vol[tuple(idx.T)].astype(np.float64)
    pts = ((idx[:, ::-1] - (np.array(vol.shape[::-1]) - 1) / 2.0) * voxel).astype(np.float32)
    c = pts[rng.choice(len(pts), n, replace=False, p=w / w.sum())].copy()
    for _ in range(iters):
        d2 = (pts ** 2).sum(1)[:, None] - 2 * pts @ c.T + (c ** 2).sum(1)[None]
        lab = d2.argmin(1)
        ws = np.bincount(lab, w, n)
        for a in range(3):
            upd = np.bincount(lab, w * pts[:, a], n)
            c[ws > 0, a] = upd[ws > 0] / ws[ws > 0]
    return c.astype(np.float32), np.bincount(lab, w, n).astype(np.float32)


def gmm_to_map(centres, amps, sigmas, box: int, voxel: float = 1.0) -> np.ndarray:
    """Density [z, y, x] (box³, origin at the box centre) of Σ_j A_j exp(−|x − c_j|² / 2σ_j²); c and σ in Å."""
    centres = np.asarray(centres, float)
    s = np.broadcast_to(np.asarray(sigmas, float), (len(centres),))
    g = (np.arange(box) - (box - 1) / 2.0) * voxel
    ex = [np.exp(-(g[None, :] - centres[:, a, None]) ** 2 / (2 * s[:, None] ** 2)) for a in range(3)]
    return np.einsum("n,nz,ny,nx->zyx", np.asarray(amps, float), ex[2], ex[1], ex[0], optimize=True).astype(np.float32)


def gmm_project(centres, amps, sigmas, R, box: int, pixel: float = 1.0) -> np.ndarray:
    """Projection [y, x] along z of a Gaussian mixture rotated by R (acting on (x, y, z)); same units as `gmm_to_map`.

    A 3D Gaussian projects to a 2D Gaussian of the same width, so the image is Σ_j w_j · g_y,j ⊗ g_x,j — two small
    matrices instead of a 3D grid (the 'project x and y separately' trick).
    """
    p = (np.asarray(centres, np.float32) @ np.asarray(R, np.float32).T)
    s = np.broadcast_to(np.asarray(sigmas, np.float32), (len(p),))[:, None]
    g = ((np.arange(box) - (box - 1) / 2.0) * pixel).astype(np.float32)[None, :]
    gx = np.exp(-(g - p[:, 0, None]) ** 2 / (2 * s ** 2))
    gy = np.exp(-(g - p[:, 1, None]) ** 2 / (2 * s ** 2))
    w = np.asarray(amps, np.float32)[:, None] * np.float32(np.sqrt(2 * np.pi)) * s / np.float32(pixel)   # ∫ dz
    return ((gy * w).T @ gx).astype(np.float32)


def reconstruct_fourier(images: np.ndarray, rotations, wiener: float = 1.0, chunk: int = 64) -> np.ndarray:
    """3D map [z, y, x] from projection images [n, y, x] with known rotations (R acts on (x, y, z), as in
    `rotate_volume`; image = rotate_volume(vol, R).sum(axis=0)).

    Projection-slice theorem: each image's 2D Fourier transform is the central slice of the 3D transform at
    k₃ = Rᵀ(kx, ky, 0). Slices are inserted with trilinear weights into a 2× padded grid, divided by the summed
    weights (+ `wiener`, in units of one image), transformed back and corrected for the interpolation kernel.
    """
    images = np.asarray(images, np.float64)
    n, N, _ = images.shape
    M, pad = 2 * N, N // 2
    k = np.arange(M) - M // 2
    ky, kx = np.meshgrid(k, k, indexing="ij")
    keep = (kx ** 2 + ky ** 2).ravel() <= (M // 2 - 1) ** 2
    plane = np.stack([kx.ravel(), ky.ravel(), np.zeros(M * M)], 1)[keep]
    shift_phase = np.exp(-1j * np.pi * (kx + ky) / M).ravel()[keep]          # half-pixel: image centre → index M/2
    Fr, Fi, W = np.zeros(M ** 3), np.zeros(M ** 3), np.zeros(M ** 3)
    corners = [(dz, dy, dx) for dz in (0, 1) for dy in (0, 1) for dx in (0, 1)]
    for start in range(0, n, chunk):
        idx_all, val_all, w_all = [], [], []
        for img, R in zip(images[start:start + chunk], list(rotations)[start:start + chunk]):
            big = np.zeros((M, M)); big[pad:pad + N, pad:pad + N] = img
            Fimg = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(big))).ravel()[keep] * shift_phase
            q = plane @ np.asarray(R, float)                                    # rows: (Rᵀ k)ᵀ in (x, y, z)
            pz = q[:, ::-1] + M // 2                                            # (z, y, x) grid coordinates
            i0 = np.floor(pz).astype(int); f = pz - i0
            for dz, dy, dx in corners:
                ii = i0 + (dz, dy, dx)
                wt = (np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1])
                      * np.where(dx, f[:, 2], 1 - f[:, 2]))
                ok = np.all((ii >= 0) & (ii < M), 1)
                idx_all.append((ii[ok, 0] * M + ii[ok, 1]) * M + ii[ok, 2]); val_all.append(wt[ok] * Fimg[ok])
                w_all.append(wt[ok])
        idx, val, wts = np.concatenate(idx_all), np.concatenate(val_all), np.concatenate(w_all)
        Fr += np.bincount(idx, val.real, M ** 3); Fi += np.bincount(idx, val.imag, M ** 3); W += np.bincount(idx, wts, M ** 3)
    F3 = ((Fr + 1j * Fi) / (W + wiener)).reshape(M, M, M)
    kz3, ky3, kx3 = np.meshgrid(k, k, k, indexing="ij")
    F3 *= np.exp(1j * np.pi * (kx3 + ky3 + kz3) / M)                            # centre back to (N − 1)/2
    vol = np.real(np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(F3))))[pad:pad + N, pad:pad + N, pad:pad + N]
    x = (np.arange(N) - (N - 1) / 2.0) / M
    sinc2 = np.sinc(x) ** 2                                                     # trilinear-kernel correction
    return (vol / (sinc2[:, None, None] * sinc2[None, :, None] * sinc2[None, None, :])).astype(np.float32)
print("🧰 toolkit loaded")

### PyTorch helpers (from Modules 25–26)

The differentiable GMM projector — now with a width per Gaussian, which Section 4 needs — the FRC, and rotation
matrices from axis-angle vectors.

In [ ]:
# ── PyTorch GMM projector, FRC and axis-angle rotations ──
N, PX = 40, 4.0
GRID_T = torch.tensor((np.arange(N) - (N - 1) / 2) * PX, dtype=torch.float32)
_fy, _fx = np.fft.fftfreq(N), np.fft.rfftfreq(N)
RING_T = torch.tensor(np.minimum((np.sqrt(_fy[:, None] ** 2 + _fx[None, :] ** 2) * N).astype(int), N // 2).ravel())


def project_gmm_torch(centres, amps, sigmas, R):
    """Projections (b, N, N) at poses R (b, 3, 3); centres (n, 3) or (b, n, 3) Å, amps (n,), sigmas (n,) Å."""
    p = torch.einsum("bij,nj->bni", R, centres) if centres.dim() == 2 else torch.einsum("bij,bnj->bni", R, centres)
    s = sigmas[None, :, None]
    gx = torch.exp(-(GRID_T - p[..., 0:1]) ** 2 / (2 * s ** 2))
    gy = torch.exp(-(GRID_T - p[..., 1:2]) ** 2 / (2 * s ** 2))
    w = (amps * math.sqrt(2 * math.pi) * sigmas / PX)[None, :, None]
    return torch.einsum("bny,bnx->byx", gy * w, gx)


def frc_torch(a, b, rmin: int, rmax: int):
    """Per-image mean Fourier ring correlation over rings rmin..rmax between batches of images (b, N, N)."""
    Fa = torch.fft.rfft2(a).reshape(a.shape[0], -1)
    Fb = torch.fft.rfft2(b).reshape(b.shape[0], -1)
    ring_sum = lambda v: torch.zeros(a.shape[0], N // 2 + 1).index_add(1, RING_T, v)
    num = ring_sum((Fa * Fb.conj()).real)
    den = torch.sqrt(ring_sum(Fa.abs() ** 2) * ring_sum(Fb.abs() ** 2) + 1e-12)
    return (num / den)[:, rmin:rmax + 1].mean(1)


def axis_angle_matrix_torch(v):
    """Rotation matrices (b, 3, 3) from axis-angle vectors v (b, 3) (direction = axis, length = angle in rad)."""
    angle = v.norm(dim=1, keepdim=True).clamp_min(1e-8)
    k = v / angle
    K = torch.zeros(v.shape[0], 3, 3)
    K[:, 0, 1], K[:, 0, 2], K[:, 1, 2] = -k[:, 2], k[:, 1], -k[:, 0]
    K[:, 1, 0], K[:, 2, 0], K[:, 2, 1] = k[:, 2], -k[:, 1], k[:, 0]
    s, c = torch.sin(angle)[..., None], torch.cos(angle)[..., None]
    return torch.eye(3)[None] + s * K + (1 - c) * K @ K


print("helpers ready")

---
## 1 · A wobbling domain

> 📍 **Where we are.** Modules 26–28 *described* heterogeneity — a latent space, a movie. This module *uses* it to get
> better maps of the moving parts. Module 30 then builds atomic models that follow the motion.

We let the **left half** of our protein — body domain 1 together with its partner, ≈ 35 kDa — wobble as a rigid body
relative to the rest, about a pivot at the interface between the two body domains. In each particle it is turned by a
small random rotation about a random axis (angle drawn from a Gaussian with σ = 15°), the way a flexible domain explores
its range in a sample. The particles' overall poses are known exactly, as if from a perfect consensus refinement of the
rigid part. 1,500 particles at SNR 0.5.

In [ ]:
# ── 1.1 The protein, the flexible region and 1,500 wobbling particles ──
atoms, hinge = flexible_protein()
Z_ATOM = np.array([{"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0}.get(e, 6.0) for e in atoms["element"]], np.float32)
FLEX = (atoms["domain"] == 0) | (atoms["domain"] == 3)                          # body 1 + partner
PIVOT = 0.5 * (atoms["xyz"][atoms["domain"] == 0].mean(0) + atoms["xyz"][atoms["domain"] == 1].mean(0))
N_PART, SNR, WOBBLE_DEG = 1500, 0.5, 15.0


def small_rotation(rng, sigma_deg: float) -> np.ndarray:
    """A rotation about a random axis by an angle drawn from N(0, sigma²) (degrees)."""
    axis = rng.normal(size=3); axis /= np.linalg.norm(axis)
    a = np.radians(rng.normal(0, sigma_deg))
    K = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(a) * K + (1 - np.cos(a)) * K @ K


def wobbled(A: np.ndarray) -> np.ndarray:
    """Atom coordinates with the flexible region turned by A about the pivot."""
    xyz = atoms["xyz"].astype(np.float64).copy()
    xyz[FLEX] = (xyz[FLEX] - PIVOT) @ A.T + PIVOT
    return xyz


A_true = np.array([small_rotation(rng, WOBBLE_DEG) for _ in range(N_PART)])        # hidden: each particle's wobble
P_true = np.array([random_rotation(rng) for _ in range(N_PART)]).astype(np.float32)   # known: overall poses
t0 = time.time()
clean = np.array([gmm_project(wobbled(A_true[i]), Z_ATOM, 2.5, P_true[i], N, PX) for i in range(N_PART)])
particles = (clean + rng.normal(0, np.sqrt(clean.var() / SNR), clean.shape)).astype(np.float32)
truth = gmm_to_map(atoms["xyz"], Z_ATOM, 2.5, N, PX)                               # the un-wobbled molecule
flex_truth = gmm_to_map(atoms["xyz"][FLEX], Z_ATOM[FLEX], 2.5, N, PX)
mask_flex = ndimage.gaussian_filter((ndimage.gaussian_filter(flex_truth, 1) > 0.1 * flex_truth.max()).astype(float), 1.5)
mask_rigid = ndimage.gaussian_filter((ndimage.gaussian_filter(truth - flex_truth, 1) > 0.1 * truth.max()).astype(float), 1.5)
angles = np.degrees(np.arccos(np.clip((np.trace(A_true, axis1=1, axis2=2) - 1) / 2, -1, 1)))
print(f"{N_PART} particles in {time.time() - t0:.0f} s; wobble angle: median {np.median(angles):.1f}°, "
      f"90th percentile {np.percentile(angles, 90):.1f}°")

---
## 2 · What the consensus does to it

We reconstruct all particles with their (correct) overall poses and judge the map **region by region**: the
real-space correlation with the true map inside the flexible region and inside the rigid region (a common way to judge
a map locally, like the map–model correlations of Module 33). We report it twice — for the map as it is, and for its
fine details only (after subtracting a 2-voxel blur, i.e. a high-pass filter), which is where blur hurts most.

In [ ]:
# ── 2.1 Consensus reconstruction and region-by-region quality ──
def region_quality(vol: np.ndarray, mask: np.ndarray) -> tuple[float, float]:
    """(correlation, fine-detail correlation) of vol with the true map inside mask (> 0.5)."""
    m = mask > 0.5
    hp = lambda v: v - ndimage.gaussian_filter(v, 2.0)
    return float(np.corrcoef(vol[m], truth[m])[0, 1]), float(np.corrcoef(hp(vol)[m], hp(truth)[m])[0, 1])


consensus = reconstruct_fourier(particles, P_true)
q_cons = {name: region_quality(consensus, m) for name, m in [("rigid", mask_rigid), ("flexible", mask_flex)]}
for name, (cc, cc_hp) in q_cons.items():
    print(f"consensus, {name:8s} region: correlation {cc:.3f}, fine-detail correlation {cc_hp:.3f}")
fig, axes = plt.subplots(1, 3, figsize=(8.5, 2.8))
axes[0].imshow(truth.sum(0)); axes[0].set_title("truth (no wobble)", fontsize=9)
axes[1].imshow(lowpass(consensus, 0.4).sum(0)); axes[1].set_title("consensus of wobbling particles", fontsize=9)
axes[2].imshow((mask_flex - mask_rigid).sum(0), cmap="coolwarm"); axes[2].set_title("regions: flexible (red), rigid (blue)", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The rigid region is reconstructed almost perfectly, the flexible region clearly less well —
above all its fine details. Nothing is wrong with the poses: the *particles themselves* disagree about where the left
half is. This is the situation of the TRPV1 cytoplasmic domains or of a rotating ATP-synthase head.

---
## 3 · A Gaussian model and a focus mask

Focused refinement needs (i) a GMM of the molecule — here K-means pseudo-atoms of the consensus plus a short FRC
refinement (Modules 25–26) — and (ii) the set $S$ of Gaussians inside the region of interest, chosen with a mask, as a
user would draw it in a map viewer.

In [ ]:
# ── 3.1 GMM of the consensus and the Gaussians of the flexible region ──
N_G, SIG_G = 200, 4.0
cen0, amp0 = pseudoatoms_kmeans(np.clip(lowpass(consensus, PX / 10), 0, None), N_G, voxel=PX, rng=np.random.default_rng(1))
X_t, P_t = torch.tensor(particles), torch.tensor(P_true)
cen_t = torch.tensor(cen0, requires_grad=True)
loga_t = torch.tensor(np.log(amp0 / amp0.max()), requires_grad=True)
opt = torch.optim.Adam([{"params": [cen_t], "lr": 0.3}, {"params": [loga_t], "lr": 0.02}])
t0 = time.time()
for it in range(120):
    idx = torch.randint(0, N_PART, (48,))
    loss = -frc_torch(project_gmm_torch(cen_t, loga_t.exp(), torch.full((N_G,), SIG_G), P_t[idx]), X_t[idx], 1, 16).mean()
    opt.zero_grad(); loss.backward(); opt.step()
gmm_c, gmm_a = cen_t.detach(), loga_t.exp().detach()
vox = np.clip(np.round(gmm_c.numpy()[:, ::-1] / PX + (N - 1) / 2).astype(int), 0, N - 1)
in_S = torch.tensor(mask_flex[tuple(vox.T)] > 0.5)
print(f"GMM fitted in {time.time() - t0:.0f} s; {int(in_S.sum())} of {N_G} Gaussians lie in the flexible region")

---
## 4 · Focused refinement

The idea of Chen (2026): give every particle $i$ its own rotation $R_i$ of the Gaussians in $S$ only,

$$M_i(\mathbf x)=\sum_{j\in S}A_j\,e^{-\lVert\mathbf x-R_i\mathbf c_j\rVert^2/2\sigma_j^2}
                +\sum_{j\notin S}A_j\,e^{-\lVert\mathbf x-\mathbf c_j\rVert^2/2\sigma_j^2},$$

*in words:* the region's Gaussians are turned as one rigid body (we turn them about the pivot), the rest stay put; then
choose $R_i$ to **maximise the FRC** between $M_i$ projected at the particle's known pose and the particle image. The
paper offers two optimisers: **direct gradients** (automatic differentiation, good for small random perturbations) and a
**neural network** in the e2gmm style — an encoder reading the FRC gradients with respect to the Gaussians, a 3D latent
space, a decoder outputting $R_i$ (Module 26's rigid-body decoder) — which suits long-range, trajectory-like motions; the
best results came from the network followed by direct refinement. We use direct gradients, for all particles at once.

Two practical ingredients:

- **Widen what is outside the focus.** Following Chen, Schmid & Chiu (2024), Gaussians outside the region are made wider,
  $\sigma'=\sigma/(0.25+0.75M)$ with $M$ the mask value (1 inside, 0 outside → 4× wider outside): the rest of the molecule
  still explains the overall image, but only at low resolution, so it does not dominate the fine comparison.
- **A gentle prior.** With noisy single images, unconstrained rotations wander. We add a small penalty on the rotation
  angle (a Gaussian prior that says "most wobbles are small") and fit only to 20 Å, where per-particle signal lives.

In [ ]:
# ── 4.1 Direct per-particle optimisation of the region's rotation ──
PIVOT_T = torch.tensor(PIVOT, dtype=torch.float32)
sig_focus = torch.where(in_S, torch.tensor(SIG_G), torch.tensor(SIG_G / 0.25))      # widen outside the focus
RMAX, PRIOR, STEPS = int(N * PX / 20), 1.0, 30


def focused_refinement(images, poses, n_particles: int, tilts_per_particle: int = 1):
    """Per-particle axis-angle rotations of the focus Gaussians that maximise the (tilt-averaged) FRC."""
    v = torch.zeros(n_particles, 3, requires_grad=True)
    opt = torch.optim.Adam([v], lr=0.01)
    for step in range(STEPS):
        for idx in torch.arange(n_particles).split(300):
            R = axis_angle_matrix_torch(v[idx])
            moved = torch.einsum("bij,nj->bni", R, gmm_c - PIVOT_T) + PIVOT_T
            cen = torch.where(in_S[None, :, None], moved, gmm_c[None].expand(len(idx), -1, -1))
            cen = cen.repeat_interleave(tilts_per_particle, 0)                 # the same motion in every tilt
            rows = (idx[:, None] * tilts_per_particle + torch.arange(tilts_per_particle)).reshape(-1)
            score = frc_torch(project_gmm_torch(cen, gmm_a, sig_focus, poses[rows]), images[rows], 1, RMAX)
            score = score.reshape(len(idx), tilts_per_particle).mean(1)        # FRC averaged over a particle's tilts
            loss = -score.sum() + PRIOR * (v[idx] ** 2).sum()
            opt.zero_grad(); loss.backward(); opt.step()
    return axis_angle_matrix_torch(v.detach()).numpy()


t0 = time.time()
R_est = focused_refinement(X_t, P_t, N_PART)
print(f"focused refinement of {N_PART} particles in {time.time() - t0:.0f} s")

flex_atoms = atoms["xyz"][FLEX].astype(np.float64)


def region_rmsd(Ra: np.ndarray, Rb: np.ndarray) -> float:
    """RMS distance (Å) between the flexible region's atoms turned by Ra and by Rb about the pivot."""
    return float(np.sqrt((((flex_atoms - PIVOT) @ Ra.T - (flex_atoms - PIVOT) @ Rb.T) ** 2).sum(1).mean()))


before = np.array([region_rmsd(np.eye(3), A) for A in A_true])
after = np.array([region_rmsd(Re, A) for Re, A in zip(R_est, A_true)])
print(f"error in placing the flexible region (median RMSD over its atoms): "
      f"before {np.median(before):.1f} Å (assuming no motion) → after {np.median(after):.1f} Å")
fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.hist(before, bins=40, alpha=0.6, label="before (no motion assumed)"); ax.hist(after, bins=40, alpha=0.6, label="after focused refinement")
ax.set_xlabel("region placement error (Å RMSD)"); ax.set_ylabel("particles"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
assert np.median(after) < np.median(before), "focused refinement should reduce the placement error"

🔍 **What just happened?** Without any knowledge of the true wobbles, the per-particle rotations of the region's ~90 Gaussians,
guided only by each noisy image's FRC, place the flexible region noticeably closer to where it really is in most
particles. The estimates are far from perfect — a 35 kDa region in a 40-pixel image at SNR 0.5 carries little
information per particle — but they need only be *better than assuming no motion* to help, as the next section shows.

✏️ **Your turn.** Set `PRIOR = 0.0` and re-run 4.1: what happens to the error? Then try `RMAX = 16` (fitting to 10 Å).

---
## 5 · From motions to poses to maps

Now the step that makes the maps better. If the region in particle $i$ is the neutral region turned by $R_i$ (about the
pivot), then, **as far as the region is concerned**, particle $i$ was simply photographed at a different pose:
projecting the neutral region with pose $P_iR_i$ (plus a small in-plane shift that accounts for the pivot) gives the
same image. (The paper writes the new pose as $P'_i=R_iP_i$ in its own convention.) We reconstruct all particles with
these **region-specific poses**: the flexible region comes into register and sharpens, while the rigid part — now
misaligned — blurs. Each map is good where its focus was, so we merge them with masks into a **composite map**,

$$\text{Composite} = \frac{\sum_r \text{Mask}_r\cdot\text{Map}_r}{\sum_r\text{Mask}_r},$$

*in words:* every voxel takes the map that was focused on its region (with smooth masks blending the borders), the
formula of Chen, Schmid & Chiu (2024).

In [ ]:
# ── 5.1 Region-specific poses → focused reconstruction → composite map ──
KX_ = np.fft.fftfreq(N)[None, :]; KY_ = np.fft.fftfreq(N)[:, None]


def realign_on_region(images, poses, R_region):
    """New poses P·R and pivot-correcting image shifts that bring the region into its neutral position."""
    shifted, new_poses = np.empty_like(images), np.empty_like(poses)
    for i, (img, P, R) in enumerate(zip(images, poses, R_region)):
        s = (P @ (PIVOT - R @ PIVOT))[:2] / PX                                    # region's image shift (pixels x, y)
        shifted[i] = np.real(np.fft.ifft2(np.fft.fft2(img) * np.exp(2j * np.pi * (KX_ * s[0] + KY_ * s[1]))))
        new_poses[i] = P @ R
    return shifted, new_poses


focused = reconstruct_fourier(*realign_on_region(particles, P_true, R_est))
oracle = reconstruct_fourier(*realign_on_region(particles, P_true, A_true))          # with the TRUE wobbles (unknowable)
composite = (mask_rigid * consensus + mask_flex * focused) / (mask_rigid + mask_flex + 1e-6)
rows = {"consensus": consensus, "focused on the region": focused, "composite": composite, "oracle (true wobbles)": oracle}
print(f"{'map':24s} {'flexible: corr / fine':>24s} {'rigid: corr / fine':>22s}")
for name, vol in rows.items():
    qf, qr = region_quality(vol, mask_flex), region_quality(vol, mask_rigid)
    print(f"{name:24s} {qf[0]:11.3f} / {qf[1]:.3f} {qr[0]:13.3f} / {qr[1]:.3f}")
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, (name, vol) in zip(axes, rows.items()):
    ax.imshow(lowpass(vol, 0.4)[N // 2]); ax.set_title(name, fontsize=9); ax.axis("off")
plt.suptitle("central slices (the flexible region is on the left)", fontsize=10)
plt.tight_layout(); plt.show()
assert region_quality(focused, mask_flex)[1] > region_quality(consensus, mask_flex)[1]

🔍 **What just happened?** Re-aligned on the region, the same particles give a clearly better flexible region —
especially its fine details — at the price of a blurred rigid part; the composite keeps the best of both. The oracle
row (true wobbles, impossible in practice) shows how much is left to gain with better per-particle estimates — which
is why the real method iterates and why bigger particles, finer data and the neural-network optimiser help.

📄 The authors report, for example, TRPV1 (EMPIAR-10059): the cytoplasmic domains improve from 2.7 Å to 2.4 Å with the
network-then-direct optimiser, and the global map from 2.8 Å (homogeneous) to 2.4 Å.

⚠️ **A caveat on global numbers.** Chen, Schmid & Chiu (2024) point out that making a flexible domain more *resolvable*
can even *lower* a global FSC (sharper but differently placed density in the half-maps of different patches). Judge
such maps with local measures — local resolution (Module 10), map–model fit and the Q-score of atom resolvability.

---
## 6 · Patches and the 2024 tricks

Chen, Schmid & Chiu (2024) start from per-particle **conformations** rather than per-particle rotations:

- **Conformation → pose conversion.** A heterogeneity network (Module 26, with a hierarchical decoder and one network per
  half-set) gives each particle a deformed GMM. For a target region, they search for a new orientation $\theta_{\text{new}}$
  that makes the *neutral* model look, inside the region's mask, like the *deformed* model at the original pose:
  minimise RMSD$\big[(\text{Proj}(\text{GMM}_{\text{neutral}}, \theta_{\text{init}}) - \text{Proj}(\text{GMM}_{\text{conf}}, \theta_{\text{new}}))\cdot\text{mask}\big]$
  (Adam, 30 iterations). Then they reconstruct with $\theta_{\text{new}}$ — our Section 5 with a different way of
  getting the rotation.
- **Patch by patch.** The molecule is cut into **K = 8 patches** by K-means on the Gaussians; each patch gets soft spherical
  masks (5-pixel fall-off), widened Gaussians outside (Section 4) and three focused iterations; the patch maps are merged
  with the composite formula and validated with a gold-standard FSC of composite half-maps.

Let us see what eight patches of our model look like.

In [ ]:
# ── 6.1 Eight patches of the GMM by K-means ──
cen_np = gmm_c.numpy()
patch_c = cen_np[np.random.default_rng(3).choice(N_G, 8, replace=False)].copy()
for _ in range(30):                                                            # plain K-means (Module 04)
    lab = np.argmin(((cen_np[:, None] - patch_c[None]) ** 2).sum(-1), 1)
    patch_c = np.array([cen_np[lab == k].mean(0) if np.any(lab == k) else patch_c[k] for k in range(8)])
print("Gaussians per patch:", np.bincount(lab, minlength=8))
fig, ax = plt.subplots(figsize=(4.4, 3.8))
ax.scatter(cen_np[:, 0], cen_np[:, 1], c=lab, cmap="tab10", s=18)
ax.scatter(patch_c[:, 0], patch_c[:, 1], marker="x", color="k", s=60)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)")
ax.set_title("8 K-means patches of the Gaussians", fontsize=9); plt.tight_layout(); plt.show()

🔍 Each colour would be refined on its own, with the others widened, and the eight maps merged. The authors report gains
such as a GPCR (EMPIAR-10786) from 3.3 Å to 2.8 Å to 2.5 Å and spike protein (EMPIAR-10492) from 3.8 Å to 3.4 Å to
3.1 Å (consensus → global GMM refinement → patch-by-patch), with Q-scores rising (0.60 → 0.67 for the GPCR); the price is
computation (about two hours per iteration for 400,000 particles on one GPU) and an empirical choice of K.

---
## 7 · Inside cells: sub-tilts

Chen (2026) extends focused refinement to cryo-ET with the sub-tilt idea of Module 28. A 3D particle $i$ with pose $T_i$
appears in tilt $k$ at pose $P_{i,k}=Q_kT_i$. The particle has **one** wobble $R_i$, so the FRC is **averaged over all
its tilts** before optimising; the new poses are $P'_{i,k}$ (in our convention $P_{i,k}R_i$), and the map is
reconstructed from the 2D sub-tilts directly — which, the paper notes, avoids the missing-wedge artefacts of averaging 3D
subtomograms. We simulate 300 particles × 5 tilts (−40° … +40°), SNR 0.5 per tilt image.

In [ ]:
# ── 7.1 Sub-tilt data, focused refinement with tilt-averaged FRC, and reconstruction from sub-tilts ──
TILTS = np.radians([-40.0, -20.0, 0.0, 20.0, 40.0])
N_ET, NT = 300, len(TILTS)
A_et = np.array([small_rotation(rng, WOBBLE_DEG) for _ in range(N_ET)])
T_et = [random_rotation(rng) for _ in range(N_ET)]
P_et = np.array([[rot_y(a) @ T for a in TILTS] for T in T_et]).astype(np.float32)          # P_ik = Q_k T_i
t0 = time.time()
clean_et = np.array([[gmm_project(wobbled(A_et[i]), Z_ATOM, 2.5, P_et[i, k], N, PX) for k in range(NT)]
                     for i in range(N_ET)])
subtilts = (clean_et + rng.normal(0, np.sqrt(clean_et.var() / SNR), clean_et.shape)).astype(np.float32)
imgs_flat, poses_flat = subtilts.reshape(-1, N, N), P_et.reshape(-1, 3, 3)
cons_et = reconstruct_fourier(imgs_flat, poses_flat)
R_et = focused_refinement(torch.tensor(imgs_flat), torch.tensor(poses_flat), N_ET, tilts_per_particle=NT)
foc_et = reconstruct_fourier(*realign_on_region(imgs_flat, poses_flat, np.repeat(R_et, NT, axis=0)))
print(f"{N_ET} particles × {NT} tilts, refined in {time.time() - t0:.0f} s")
print(f"region placement error: before {np.median([region_rmsd(np.eye(3), A) for A in A_et]):.1f} Å → after "
      f"{np.median([region_rmsd(Re, A) for Re, A in zip(R_et, A_et)]):.1f} Å")
for name, vol in [("sub-tilt consensus", cons_et), ("sub-tilt focused", foc_et)]:
    qf = region_quality(vol, mask_flex)
    print(f"{name:20s}: flexible region correlation {qf[0]:.3f}, fine detail {qf[1]:.3f}")

🔍 **What just happened?** With only 300 particles — but five views of each — the tilt-averaged FRC pins down each
particle's wobble even a little better than the 1,500 single images of Section 4 did, and re-aligning the sub-tilts on the region sharpens it in the
same way. This is how the paper treats a rotary motor *in situ*: for ATP synthase in *Chlamydomonas reinhardtii* cells
(EMPIAR-11830) the authors report a global map at 5.6 Å, the peripheral stalk improving from 5.4 to 5.2 Å, the F1 head —
which rotates by about 25° — from about 8 Å to 6.5 Å, and a composite map at 5.2 Å.

### 📄 Paper connection — Chen 2026: GMM-based focused refinement in cryo-EM and cryo-ET

**Paper:** Chen M. *Gaussian mixture model-based focused refinement for enhanced flexible structure determination in
CryoEM and CryoET.* arXiv:2605.30518 (2026). <https://arxiv.org/abs/2605.30518>

- **Question it answers:** how to get high-resolution maps of flexible regions when masked, region-only alignment of noisy
  particles is unreliable.
- **What the paper actually does:** rotates only the Gaussians of a selected set $S$ by a per-particle $R_i$ and maximises
  the FRC of the projection at the particle's pose; sets $P'_i=R_iP_i$; reconstructs; iterates, with separate half-sets.
  Optimisers: direct gradients (JAX automatic differentiation) for small random perturbations; a network (encoder input
  = per-Gaussian $\partial\mathrm{FRC}/\partial\mathbf c_j$, 3D latent, decoder outputs $R_i$) for long-range motions;
  best: network then direct. With a whole-protein mask the method reduces to global GMM refinement. For cryo-ET it works
  on 2D sub-tilts with $P_{i,k}=Q_kT_i$, one $R_i$ per 3D particle, the FRC averaged over its tilts, and reconstruction
  from the 2D particles.
- **Result reported:** TRPV1 (EMPIAR-10059) 2.8 Å homogeneous → 2.4 Å with global GMM refinement; cytoplasmic domain
  2.7 → 2.4 Å; ATP synthase in cells (EMPIAR-11830): global 5.6 Å, peripheral stalk 5.4 → 5.2 Å, F1 head ≈ 8 → 6.5 Å,
  composite 5.2 Å; on simulated data the network and direct optimisers gave similar resolutions (e.g. 3.48 vs 3.55 Å).
- **What our toy keeps:** focus set $S$, per-particle rigid rotation by direct gradients on the FRC, new poses, focused
  reconstruction, composite, the sub-tilt version with tilt-averaged FRC.
- **What our toy leaves out:** the network optimiser, iterations and half-sets, CTF, high resolution.

### 📄 Paper connection — Chen, Schmid & Chiu 2024: resolution and resolvability with GMMs

**Paper:** Chen M, Schmid MF, Chiu W. *Improving resolution and resolvability of single-particle cryoEM structures using
Gaussian mixture models.* Nature Methods 21:37–40 (2024). [doi:10.1038/s41592-023-02082-9](https://doi.org/10.1038/s41592-023-02082-9)

- **What the paper actually does (the part used here):** a heterogeneity network with a hierarchical 32-Gaussian decoder,
  pre-trained so that $\mathbf z=0$ gives the consensus, one network per half-set; conversion of each particle's
  conformation into a new pose for a target region (masked projection RMSD, Adam, 30 iterations); patch-by-patch
  refinement with K = 8 K-means patches, widening $\sigma'=\sigma/(0.25+0.75M)$ outside the patch, soft spherical masks
  (5-pixel fall-off) and three iterations per patch; composite map $\sum\text{Mask}_i\text{Map}_i/\sum\text{Mask}_i$ and
  gold-standard FSC of composite half-maps.
- **Result reported:** GPCR (EMPIAR-10786) 3.3 → 2.8 → 2.5 Å, Q-score 0.60 → 0.67; spike (EMPIAR-10492) 3.8 → 3.4 →
  3.1 Å, Q 0.47 → 0.53; ABC transporter (EMPIAR-10374) 3.2 → 2.8 → 2.5 Å; with 6,000-particle subsets the GPCR went from
  4.5 to 4.0 to 3.9 Å and the spike from 6.2 to 5.3 to 4.6 Å.
- **Limitations stated:** about two hours per iteration for 400,000 particles (one A5000 GPU); K chosen empirically; better
  resolvability of flexible domains can lower the global FSC, so local resolution and Q-scores should be used.

---
## ✅ Checkpoint

1. All particles have correct overall poses, yet one region of the consensus is blurred. Why?
   <details><summary>Answer</summary>The region itself sits differently in different particles; averaging misplaced copies blurs it (Module 04: averaging works only for aligned copies).</details>
2. What exactly is optimised in GMM-based focused refinement, and against what?
   <details><summary>Answer</summary>A rotation per particle applied only to the Gaussians of the focus set S, chosen to maximise the FRC between the projected model and that particle's image at its known pose.</details>
3. Why widen the Gaussians outside the focus region?
   <details><summary>Answer</summary>So that the rest of the molecule still accounts for the image at low resolution but does not dominate the fine comparison that determines the region's motion.</details>
4. How do per-particle rotations of a region turn into a sharper map?
   <details><summary>Answer</summary>They define, for that region, a new pose for every particle (P·R plus a pivot shift); reconstructing with those poses brings the region into register.</details>
5. In cryo-ET, why average the FRC over a particle's tilts before optimising?
   <details><summary>Answer</summary>All tilts show the same molecule in the same conformation, so they must share one R_i; pooling their evidence makes the estimate far less noisy.</details>
6. Why can a better map of a flexible domain lower the global FSC?
   <details><summary>Answer</summary>Global FSC averages over the whole box; differently focused half-maps and the redistribution of density can lower some shells even when the domain becomes more interpretable. Local measures are needed.</details>

## 🧠 Questions & discussion

1. Our wobble was a rigid rotation. What if the region also *deforms* internally? Which parts of Section 4 would you change?
2. How would you choose the pivot in a real protein? What if you put it in the wrong place?
3. The prior assumed small wobbles. For ATP synthase's ≈ 25° rotary motion, why does the paper prefer the network optimiser?
4. Classical focused refinement aligns masked particle images to a masked reference. What does the GMM version know that
   the masked version does not?
5. How would you validate a composite map honestly, with half-sets, from start to finish?

## 🚀 Bonus — going further

- **Iterate.** Build a new GMM from the composite map and run Sections 3–5 again. Does a second round help in our toy?
- **Two regions.** Make the arm (domain 2) wobble too, independently, and refine two focus sets in turn; merge three maps.
- **Half-sets.** Split the particles in two from the start, run everything twice, and compute the FSC between the two
  composite maps (Module 10).
- **Real software:** EMAN2's GMM programs (<https://github.com/cryoem/eman2>) implement the papers' workflows.

## 📝 Lab note

```text
Module 29 — date:
Input: 1,500 particles (left half wobbling, σ = 15°), SNR 0.5, known poses; 300 particles × 5 sub-tilts
Output: consensus, per-particle region rotations, focused and composite maps, region-wise correlations
Method: GMM focus set, direct gradient on the FRC with widening and a prior, region-specific poses, composite, sub-tilts
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Chen 2026 (arXiv:2605.30518); Chen, Schmid & Chiu 2024
```

## 📚 References

1. Chen M. Gaussian mixture model-based focused refinement for enhanced flexible structure determination in CryoEM and CryoET. arXiv:2605.30518 (2026).
2. Chen M, Schmid MF, Chiu W. Improving resolution and resolvability of single-particle cryoEM structures using Gaussian mixture models. *Nat Methods* 21:37–40 (2024). doi:10.1038/s41592-023-02082-9
3. Chen M, Ludtke SJ. Deep learning-based mixed-dimensional Gaussian mixture model for characterizing variability in cryo-EM. *Nat Methods* 18:930–936 (2021). doi:10.1038/s41592-021-01220-5
4. Pintilie G *et al.* Measurement of atom resolvability in cryo-EM maps with Q-scores. *Nat Methods* 17:328–334 (2020). doi:10.1038/s41592-020-0731-1
5. Punjani A, Fleet DJ. 3DFlex: determining structure and motion of flexible proteins from cryo-EM. *Nat Methods* 20:860–870 (2023). doi:10.1038/s41592-023-01853-8

## ➡️ Next

**[30 · From maps to moving atoms: molecular model series with GMMs and neural networks](30_atomic_models_model_series.ipynb)**
— one Gaussian per atom, chemistry as a restraint, and a whole series of atomic models along a motion.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")